In [1]:
"""
Coverage Diagnostics & Similarity Distribution Analysis
========================================================
Tasks:
  1. Coverage of top30/top50/top100 choice sets (product count & sales share)
  2. Similarity distribution of secondary products (count & share-weighted)

Input files (in working directory):
  - full_data.dta
  - full_product_similarity.dta

Output: figures + summary tables in diagnostics/ folder
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
WORK_DIR = r"G:\Kuangyu_Temp\Outsource"
OUT_DIR = os.path.join(WORK_DIR, "diagnostics")
os.makedirs(OUT_DIR, exist_ok=True)
os.chdir(WORK_DIR)

TOP_N_LIST = [30, 50, 100]
NBINS = 50  # number of bins for histograms

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})


# ============================================================
# PART 0: Load data
# ============================================================
print("=" * 60)
print("Loading data...")
print("=" * 60)

# --- Load full_data ---
print("  Reading full_data.dta ...")
df = pd.read_stata("full_data.dta",
                    columns=['firm_id', 'year', 'product_id',
                             'is_intermediary', 'production_value',
                             'total_output', 'outsourcing_percen'])
print(f"  full_data: {len(df):,} rows")

# Drop intermediaries
df = df[df['is_intermediary'] != 1].copy()
df.drop(columns=['is_intermediary'], inplace=True)
print(f"  After dropping intermediaries: {len(df):,} rows")

# --- Define main product = max production_value per firm×year ---
df['prod_rank'] = df.groupby(['firm_id', 'year'])['production_value'] \
                    .rank(method='first', ascending=False)
df['is_main'] = (df['prod_rank'] == 1).astype(int)

# --- Separate main and secondary ---
main_df = df[df['is_main'] == 1][['firm_id', 'year', 'product_id']].copy()
main_df.rename(columns={'product_id': 'main_pid'}, inplace=True)

sec_df = df[df['is_main'] == 0].copy()
sec_df = sec_df.merge(main_df, on=['firm_id', 'year'], how='left')
# total_output here is the secondary product's sales
sec_df.rename(columns={'total_output': 'sec_sales'}, inplace=True)

# Compute total secondary sales per firm×year
sec_df['total_sec_sales'] = sec_df.groupby(['firm_id', 'year'])['sec_sales'] \
                                  .transform('sum')
sec_df['sec_sales_share'] = sec_df['sec_sales'] / sec_df['total_sec_sales']
sec_df['sec_sales_share'] = sec_df['sec_sales_share'].fillna(0)

print(f"  Main products: {len(main_df):,}")
print(f"  Secondary products: {len(sec_df):,}")

del df  # free memory

# --- Load similarity matrix ---
print("  Reading full_product_similarity.dta ...")
sim_df = pd.read_stata("full_product_similarity.dta")
print(f"  Similarity pairs (unidirectional): {len(sim_df):,}")

# Make bidirectional
sim_rev = sim_df.rename(columns={'product_1': 'product_2', 'product_2': 'product_1'})
sim_bi = pd.concat([sim_df, sim_rev], ignore_index=True)
sim_bi.drop_duplicates(subset=['product_1', 'product_2'], inplace=True)
sim_bi.rename(columns={'product_1': 'main_pid', 'product_2': 'product_id'},
              inplace=True)
print(f"  Bidirectional pairs: {len(sim_bi):,}")

del sim_df, sim_rev


# ============================================================
# PART 1: Merge similarity to secondary products
# ============================================================
print("\n" + "=" * 60)
print("Merging similarity to secondary products...")
print("=" * 60)

sec_df['main_pid'] = sec_df['main_pid'].astype(str).str.strip()
sec_df['product_id'] = sec_df['product_id'].astype(str).str.strip()
sim_bi['main_pid'] = sim_bi['main_pid'].astype(str).str.strip()
sim_bi['product_id'] = sim_bi['product_id'].astype(str).str.strip()

sec_with_sim = sec_df.merge(
    sim_bi[['main_pid', 'product_id', 'input_similarity', 'output_similarity']],
    on=['main_pid', 'product_id'],
    how='left'
)

n_total = len(sec_with_sim)
n_matched = sec_with_sim['input_similarity'].notna().sum()
print(f"  Total secondary products: {n_total:,}")
print(f"  Matched with similarity: {n_matched:,} ({n_matched/n_total*100:.1f}%)")
print(f"  Unmatched (same as main or missing): {n_total - n_matched:,}")

# Drop unmatched (product_id == main_pid or not in similarity matrix)
sec_with_sim = sec_with_sim.dropna(subset=['input_similarity'])
print(f"  Working sample: {len(sec_with_sim):,}")


# ============================================================
# PART 2: Build choice sets for top30/50/100 and compute coverage
# ============================================================
print("\n" + "=" * 60)
print("Building choice sets and computing coverage...")
print("=" * 60)

# Rank products by input_sim and output_sim for each main product
print("  Ranking products per main product...")
ranks = sim_bi.copy()
ranks['rank_input'] = ranks.groupby('main_pid')['input_similarity'] \
                           .rank(method='first', ascending=False)
ranks['rank_output'] = ranks.groupby('main_pid')['output_similarity'] \
                            .rank(method='first', ascending=False)

coverage_results = []

for top_n in TOP_N_LIST:
    print(f"\n  --- Top {top_n} ---")

    # Choice set = input top N ∪ output top N
    choice = ranks[(ranks['rank_input'] <= top_n) |
                   (ranks['rank_output'] <= top_n)].copy()
    choice = choice[['main_pid', 'product_id']].drop_duplicates()

    n_pairs = len(choice)
    avg_candidates = choice.groupby('main_pid').size().mean()
    print(f"    Total pairs: {n_pairs:,}, avg candidates/main: {avg_candidates:.1f}")

    # Mark which secondary products are in the choice set
    choice['in_topN'] = 1
    merged = sec_with_sim.merge(
        choice, on=['main_pid', 'product_id'], how='left'
    )
    merged['in_topN'] = merged['in_topN'].fillna(0).astype(int)

    # --- Coverage by count ---
    n_covered = merged['in_topN'].sum()
    n_all = len(merged)
    count_coverage = n_covered / n_all

    # --- Coverage by secondary sales share ---
    # For each firm×year, what share of secondary sales is covered?
    covered_sales = merged.loc[merged['in_topN'] == 1, 'sec_sales'].sum()
    total_sales = merged['sec_sales'].sum()
    sales_coverage = covered_sales / total_sales

    print(f"    Product count coverage: {count_coverage*100:.1f}%")
    print(f"    Sales share coverage:   {sales_coverage*100:.1f}%")

    coverage_results.append({
        'top_N': top_n,
        'n_pairs': n_pairs,
        'avg_candidates': avg_candidates,
        'count_coverage': count_coverage,
        'sales_coverage': sales_coverage,
    })

    del choice, merged

# Save coverage summary
cov_df = pd.DataFrame(coverage_results)
cov_df.to_csv(os.path.join(OUT_DIR, "coverage_summary.csv"), index=False)
print(f"\n  Coverage summary saved to diagnostics/coverage_summary.csv")
print(cov_df.to_string(index=False))

del ranks


# ============================================================
# PART 3: Similarity distributions of secondary products
# ============================================================
print("\n" + "=" * 60)
print("Plotting similarity distributions...")
print("=" * 60)

for sim_var, sim_label in [('input_similarity', 'Input Similarity'),
                           ('output_similarity', 'Output Similarity')]:

    vals = sec_with_sim[sim_var].values
    weights = sec_with_sim['sec_sales_share'].values
    sales = sec_with_sim['sec_sales'].values

    # Normalize sales weights to sum to 1
    sales_total = sales.sum()
    sales_weights = sales / sales_total if sales_total > 0 else sales

    # ----------------------------------------------------------
    # Figure A: Count (unweighted) histogram
    # ----------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(vals, bins=NBINS, color='steelblue', edgecolor='white',
            alpha=0.85)
    ax.set_xlabel(sim_label)
    ax.set_ylabel('Number of Secondary Products')
    ax.set_title(f'Distribution of {sim_label} — Unweighted (Count)')
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(0, 0))
    fig.tight_layout()
    fname = f"dist_{sim_var}_count.png"
    fig.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close(fig)
    print(f"  Saved {fname}")

    # ----------------------------------------------------------
    # Figure B: Weighted by within-firm secondary sales share
    #   (each product's share of its firm×year's total secondary sales)
    # ----------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(vals, bins=NBINS, weights=weights,
            color='darkorange', edgecolor='white', alpha=0.85)
    ax.set_xlabel(sim_label)
    ax.set_ylabel('Cumulative Within-Firm Secondary Sales Share')
    ax.set_title(f'Distribution of {sim_label} — Weighted by Secondary Sales Share')
    fig.tight_layout()
    fname = f"dist_{sim_var}_share_within.png"
    fig.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close(fig)
    print(f"  Saved {fname}")

    # ----------------------------------------------------------
    # Figure C: Weighted by global secondary sales
    #   (each product's sales / total secondary sales across all firms)
    # ----------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(vals, bins=NBINS, weights=sales_weights,
            color='forestgreen', edgecolor='white', alpha=0.85)
    ax.set_xlabel(sim_label)
    ax.set_ylabel('Share of Total Secondary Sales')
    ax.set_title(f'Distribution of {sim_label} — Weighted by Total Secondary Sales')
    fig.tight_layout()
    fname = f"dist_{sim_var}_share_global.png"
    fig.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close(fig)
    print(f"  Saved {fname}")

    # ----------------------------------------------------------
    # Figure D: Overlay — count vs share-weighted (normalized to density)
    # ----------------------------------------------------------
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Count density (left axis)
    ax1.hist(vals, bins=NBINS, density=True,
             color='steelblue', edgecolor='white', alpha=0.5,
             label='Count (density)')
    ax1.set_xlabel(sim_label)
    ax1.set_ylabel('Density (Count)', color='steelblue')
    ax1.tick_params(axis='y', labelcolor='steelblue')

    # Share-weighted density (right axis)
    ax2 = ax1.twinx()
    ax2.hist(vals, bins=NBINS, weights=weights, density=True,
             color='darkorange', edgecolor='white', alpha=0.5,
             label='Sales-Share Weighted (density)')
    ax2.set_ylabel('Density (Sales-Share Weighted)', color='darkorange')
    ax2.tick_params(axis='y', labelcolor='darkorange')

    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    ax1.set_title(f'{sim_label}: Count vs Sales-Share Weighted Density')
    fig.tight_layout()
    fname = f"dist_{sim_var}_overlay.png"
    fig.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close(fig)
    print(f"  Saved {fname}")


# ============================================================
# PART 4: Coverage by similarity bins (for visual diagnostics)
# ============================================================
print("\n" + "=" * 60)
print("Coverage by similarity bins...")
print("=" * 60)

for sim_var, sim_label in [('input_similarity', 'Input Similarity'),
                           ('output_similarity', 'Output Similarity')]:

    # Re-merge with top N flags
    for top_n in TOP_N_LIST:
        choice = sim_bi.copy()
        choice['rank_input'] = choice.groupby('main_pid')['input_similarity'] \
                                     .rank(method='first', ascending=False)
        choice['rank_output'] = choice.groupby('main_pid')['output_similarity'] \
                                      .rank(method='first', ascending=False)
        choice = choice[(choice['rank_input'] <= top_n) |
                        (choice['rank_output'] <= top_n)]
        choice = choice[['main_pid', 'product_id']].drop_duplicates()
        choice[f'in_top{top_n}'] = 1
        sec_with_sim = sec_with_sim.merge(
            choice, on=['main_pid', 'product_id'], how='left'
        )
        sec_with_sim[f'in_top{top_n}'] = sec_with_sim[f'in_top{top_n}'].fillna(0)
        del choice

    # Bin by similarity
    sec_with_sim[f'{sim_var}_bin'] = pd.cut(
        sec_with_sim[sim_var], bins=20, include_lowest=True
    )

    bin_stats = []
    for bn, grp in sec_with_sim.groupby(f'{sim_var}_bin', observed=True):
        row = {
            'bin': str(bn),
            'bin_mid': bn.mid,
            'count': len(grp),
            'total_sales': grp['sec_sales'].sum(),
        }
        for top_n in TOP_N_LIST:
            covered = grp[f'in_top{top_n}']
            row[f'count_cov_top{top_n}'] = covered.sum() / len(grp) if len(grp) > 0 else 0
            covered_sales = grp.loc[covered == 1, 'sec_sales'].sum()
            row[f'sales_cov_top{top_n}'] = covered_sales / grp['sec_sales'].sum() \
                if grp['sec_sales'].sum() > 0 else 0
        bin_stats.append(row)

    bin_df = pd.DataFrame(bin_stats)

    # Plot coverage rate by similarity bin
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    for top_n in TOP_N_LIST:
        ax1.plot(bin_df['bin_mid'], bin_df[f'count_cov_top{top_n}'],
                 marker='o', markersize=4, label=f'Top {top_n}')
        ax2.plot(bin_df['bin_mid'], bin_df[f'sales_cov_top{top_n}'],
                 marker='o', markersize=4, label=f'Top {top_n}')

    ax1.set_xlabel(sim_label)
    ax1.set_ylabel('Product Count Coverage Rate')
    ax1.set_title(f'Count Coverage by {sim_label} Bin')
    ax1.legend()
    ax1.set_ylim(-0.05, 1.05)

    ax2.set_xlabel(sim_label)
    ax2.set_ylabel('Sales Share Coverage Rate')
    ax2.set_title(f'Sales Coverage by {sim_label} Bin')
    ax2.legend()
    ax2.set_ylim(-0.05, 1.05)

    fig.tight_layout()
    fname = f"coverage_by_{sim_var}_bin.png"
    fig.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close(fig)
    print(f"  Saved {fname}")

    # Drop temp columns
    for top_n in TOP_N_LIST:
        sec_with_sim.drop(columns=[f'in_top{top_n}'], inplace=True, errors='ignore')
    sec_with_sim.drop(columns=[f'{sim_var}_bin'], inplace=True, errors='ignore')


# ============================================================
# Summary
# ============================================================
print("\n" + "=" * 60)
print("All done! Output in diagnostics/ folder:")
print("=" * 60)
print("  coverage_summary.csv")
print("  dist_input_similarity_count.png")
print("  dist_input_similarity_share_within.png")
print("  dist_input_similarity_share_global.png")
print("  dist_input_similarity_overlay.png")
print("  dist_output_similarity_count.png")
print("  dist_output_similarity_share_within.png")
print("  dist_output_similarity_share_global.png")
print("  dist_output_similarity_overlay.png")
print("  coverage_by_input_similarity_bin.png")
print("  coverage_by_output_similarity_bin.png")

Loading data...
  Reading full_data.dta ...
  full_data: 90,296,650 rows
  After dropping intermediaries: 87,432,386 rows
  Main products: 11,569,923
  Secondary products: 75,862,463
  Reading full_product_similarity.dta ...
  Similarity pairs (unidirectional): 3,857,253
  Bidirectional pairs: 7,714,506

Merging similarity to secondary products...
  Total secondary products: 75,862,463
  Matched with similarity: 75,862,463 (100.0%)
  Unmatched (same as main or missing): 0
  Working sample: 75,862,463

Building choice sets and computing coverage...
  Ranking products per main product...

  --- Top 30 ---
    Total pairs: 148,584, avg candidates/main: 53.5
    Product count coverage: 24.4%
    Sales share coverage:   58.0%

  --- Top 50 ---
    Total pairs: 244,761, avg candidates/main: 88.1
    Product count coverage: 30.9%
    Sales share coverage:   65.8%

  --- Top 100 ---
    Total pairs: 480,224, avg candidates/main: 172.9
    Product count coverage: 41.7%
    Sales share coverage: